In [0]:
%run ./00_Config_and_Logging

In [0]:
# ==========================================
# 01_Raw_to_Bronze: explicit schema (schema-on-read, no inferSchema)
# ==========================================
import re
import json
from datetime import date, datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType, ArrayType)

hourly_units_schema = StructType([
    StructField("time", StringType(), True),
    StructField("carbon_monoxide", StringType(), True),
    StructField("carbon_dioxide", StringType(), True),
    StructField("pm2_5", StringType(), True),
    StructField("pm10", StringType(), True),
    StructField("nitrogen_dioxide", StringType(), True)
])

hourly_schema = StructType([
    StructField("time", ArrayType(StringType()), True),
    StructField("carbon_monoxide", ArrayType(DoubleType()), True),
    StructField("carbon_dioxide", ArrayType(DoubleType()), True),
    StructField("pm2_5", ArrayType(DoubleType()), True),
    StructField("pm10", ArrayType(DoubleType()), True),
    StructField("nitrogen_dioxide", ArrayType(DoubleType()), True)
])

open_meteo_schema = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("generationtime_ms", DoubleType(), True),
    StructField("utc_offset_seconds", IntegerType(), True),
    StructField("timezone", StringType(), True),
    StructField("timezone_abbreviation", StringType(), True),
    StructField("elevation", DoubleType(), True),
    StructField("hourly_units", hourly_units_schema, True),
    StructField("hourly", hourly_schema, True),
    StructField("city", StringType(), True),
    StructField("source_system", StringType(), True)
])
print("Explicit schema defined.")

In [0]:
# ==========================================
# Ingest ONE file into Bronze.
#
# Design: Bronze is append-only (an audit trail of what arrived and when).
# Re-loading a file adds raw rows again; Silver's MERGE INTO is what
# guarantees no duplicates downstream.
#
# Schema drift: the file is read with the explicit schema PLUS Databricks'
# rescuedDataColumn. Anything that does not fit the schema -- a column the
# source added, or a value whose type changed -- lands in _rescued_data
# instead of being silently dropped or crashing the read. Conforming rows
# go to Bronze; non-conforming rows go to the quarantine table, and the
# rest of the batch finishes normally.
# ==========================================
QUARANTINE_REASON = "SCHEMA_DRIFT: unexpected column or data-type mismatch"

def ingest_raw_to_bronze(input_file_path, bronze_table_path, quarantine_table_path,
                         load_type="incremental"):
    layer = "Raw-to-Bronze"
    start_time = datetime.now()
    status, error, n_good, n_bad = "Failure", None, 0, 0
    print(f"--- {layer} [{load_type}]: {input_file_path} ---")

    try:
        load_ts = datetime.now()   # one identical load_timestamp for the whole file
        raw_df = (spark.read
                  .schema(open_meteo_schema)
                  .option("multiline", "true")
                  .option("rescuedDataColumn", "_rescued_data")
                  .json(input_file_path)
                  .withColumn("source_file", F.col("_metadata.file_path"))
                  .withColumn("load_timestamp", F.lit(load_ts)))

        counts = raw_df.agg(
            F.sum(F.when(F.col("_rescued_data").isNull(), 1).otherwise(0)).alias("good"),
            F.sum(F.when(F.col("_rescued_data").isNotNull(), 1).otherwise(0)).alias("bad")
        ).first()
        n_good, n_bad = int(counts["good"] or 0), int(counts["bad"] or 0)

        if n_good > 0:
            (raw_df.filter(F.col("_rescued_data").isNull()).drop("_rescued_data")
                   .write.format("delta").mode("append")
                   .option("mergeSchema", "true").save(bronze_table_path))

        if n_bad > 0:
            (raw_df.filter(F.col("_rescued_data").isNotNull())
                   .select("city", "source_file",
                           F.col("_rescued_data").alias("rescued_data"),
                           F.lit(QUARANTINE_REASON).alias("quarantine_reason"),
                           "load_timestamp")
                   .write.format("delta").mode("append")
                   .option("mergeSchema", "true").save(quarantine_table_path))

        if n_bad > 0:
            status = "Partial Success"
            error = (f"{n_bad} of {n_good + n_bad} rows quarantined ({QUARANTINE_REASON}); "
                     f"see bronze_quarantine for source_file {input_file_path}")
        else:
            status = "Success"
        print(f"{status}: {n_good} rows to Bronze, {n_bad} rows quarantined.")

    except Exception as e:
        error = str(e)[:1000]
        print(f"FAILED: {error}")

    finally:
        write_log(layer, input_file_path, load_type, start_time, status,
                  rows_inserted=n_good, rows_updated=0, rows_quarantined=n_bad,
                  error_message=error)
    return status

In [0]:
# ==========================================
# Parameterized runner: incremental load AND backfill from the same code
#
#   mode="incremental"  -> every file waiting in staging/new
#   mode="backfill"     -> files for a DATE RANGE, searched in staging/new
#                          AND staging/processed (so already-loaded days can be replayed)
#
# A file's date comes from its name: incremental_YYYY-MM-DD.json.
# File that loaded successfully from new/ is moved to processed/;
# a file that fails stays where it is.
# ==========================================
def _file_date(name):
    m = re.search(r"(\d{4}-\d{2}-\d{2})", name)
    return date.fromisoformat(m.group(1)) if m else None


def list_staged_files(include_processed=False, start_date=None, end_date=None, file_name=None):
    s = date.fromisoformat(start_date) if start_date else None
    e = date.fromisoformat(end_date) if end_date else s      # only start given -> that one day
    folders = [NEW_DIR] + ([PROCESSED_DIR] if include_processed else [])
    found = []
    for folder in folders:
        for name in os.listdir(folder):
            if not name.endswith(".json"):
                continue
            if file_name:
                if name != file_name:
                    continue
            elif s or e:
                d = _file_date(name)
                if d is None or (s and d < s) or (e and d > e):
                    continue
            found.append((name, f"{folder}/{name}", folder == NEW_DIR))
    return sorted(found, key=lambda x: x[0])


def move_to_processed(src_path):
    name = os.path.basename(src_path)
    dst = f"{PROCESSED_DIR}/{name}"
    if os.path.exists(dst):   # same day re-fetched: keep both, never overwrite history
        stem, ext = os.path.splitext(name)
        dst = f"{PROCESSED_DIR}/{stem}__{datetime.now():%Y%m%dT%H%M%S}{ext}"
    dbutils.fs.mv(src_path, dst)
    return dst


def run_raw_to_bronze(mode="incremental", start_date=None, end_date=None, file_name=None,
                      bronze_table_path=None, quarantine_table_path=None):
    if mode not in ("incremental", "backfill"):
        raise ValueError("mode must be 'incremental' or 'backfill'")
    bronze_table_path = bronze_table_path or bronze_delta_path
    quarantine_table_path = quarantine_table_path or quarantine_path

    files = list_staged_files(include_processed=(mode == "backfill"),
                              start_date=start_date, end_date=end_date, file_name=file_name)
    if not files:
        print("No matching files found.")
        return []

    results = []
    for name, path, in_new in files:
        load_type = "full" if name.startswith("full_load") else mode
        status = ingest_raw_to_bronze(path, bronze_table_path, quarantine_table_path, load_type)
        if status in ("Success", "Partial Success") and in_new:
            print("Moved to:", move_to_processed(path))
        results.append((name, status))
    return results

In [0]:
# ==========================================
# RUN: set the widgets at the top of the notebook (or pass them as job parameters)
#   Daily load     : mode=incremental, leave the rest empty
#   Backfill       : mode=backfill, start_date=2026-10-01, end_date=2026-10-05
#   One exact file : mode=backfill, file_name=full_load_last_3_years.json
# ==========================================
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("file_name", "")

results = run_raw_to_bronze(
    mode=dbutils.widgets.get("mode"),
    start_date=dbutils.widgets.get("start_date") or None,
    end_date=dbutils.widgets.get("end_date") or None,
    file_name=dbutils.widgets.get("file_name") or None,
)
for r in results:
    print(r)

In [0]:
# ==========================================
# Evidence: Bronze, quarantine and the audit log
# ==========================================
bronze = spark.read.format("delta").load(bronze_delta_path)
print("Bronze rows (one row per city per file):", bronze.count())
display(bronze.select("city", "source_file", "load_timestamp").orderBy(F.col("load_timestamp").desc()).limit(10))

display(spark.read.format("delta").load(log_table_path).orderBy(F.col("start_time").desc()))

In [0]:
# ==========================================
# DEMO: schema drift handling (isolated test paths, does not touch real Bronze)
# Three records: one valid, one with a NEW column, one with a CHANGED data type.
# Expected: 1 row in test Bronze, 2 rows in test quarantine, batch status = Success.
# ==========================================
test_dir = f"{STAGING_ROOT}/_drift_test"
dbutils.fs.rm(test_dir, recurse=True)
dbutils.fs.mkdirs(test_dir)

base = {"latitude": 31.5, "longitude": 74.3, "timezone": "Asia/Karachi",
        "hourly": {"time": ["2026-10-01T00:00"], "carbon_dioxide": [415.0]},
        "source_system": "drift-test"}
records = [
    {**base, "city": "Drift_GoodRecord"},
    {**base, "city": "Drift_NewColumn", "uv_index": 7.2},               # source added a column
    {**base, "city": "Drift_TypeChange", "latitude": "not-a-number"},   # source changed a type
]
test_file = f"{test_dir}/drift_test.json"
with open(test_file, "w") as f:
    json.dump(records, f)

ingest_raw_to_bronze(test_file, f"{test_dir}/bronze", f"{test_dir}/quarantine", load_type="drift_test")

print("TEST BRONZE (expect 1 row):")
display(spark.read.format("delta").load(f"{test_dir}/bronze").select("city", "latitude", "load_timestamp"))
print("TEST QUARANTINE (expect 2 rows):")
display(spark.read.format("delta").load(f"{test_dir}/quarantine"))